# 01 · Quick start: build geodesic domes and inspect the data structure

A `GeodesicDome` is an icosahedron whose 20 triangles are each cut into *f*² smaller ones and pushed out
onto the unit sphere. This notebook builds domes, exports them as NumPy arrays, looks at individual
vertices and shows how the dome grows with the frequency *f*.

Script version: [`examples/01_quickstart.py`](../01_quickstart.py) · Needs: numpy, ipywidgets (optional: `ipympl` for live, rotatable figures)

<sub>SPDX-License-Identifier: AGPL-3.0-or-later · Copyright (C) 2022-2026 Masahiro Takatsuka. See the NOTICE file for attribution terms.</sub>

## Set-up

In [1]:
import nb_setup  # makes mt.geodesicdome and examples/dome_utils.py importable from a checkout

WIDGET = nb_setup.setup()  # live figures with ipympl, static images otherwise

figures: live (ipympl) -- drag 3D plots to rotate them


In [2]:
import numpy as np
from dome_utils import unique_mesh
from ipywidgets import IntSlider, interact

from mt.geodesicdome.grid.geodesicdome import GeodesicDome

## 1. Build a dome

`frequency` = the number of segments each icosahedron edge is cut into.

In [3]:
dome = GeodesicDome(frequency=4)
dome

GeodesicDome(frequency=4)

`split(n)` refines an existing dome and is cumulative, so two `split(2)` calls give frequency 4:

In [4]:
dome_b = GeodesicDome()          # plain icosahedron, frequency 1
dome_b.split(2)
dome_b.split(2)
assert dome_b.frequency == dome.frequency == 4
dome_b

GeodesicDome(frequency=4)

## 2. The geometry as NumPy arrays

`get_all_xyz()` returns one row per **stored** vertex and `get_all_triangles()` the vertex ids of every
triangle, wound counter-clockwise when seen from outside.

In [5]:
xyz = dome.get_all_xyz()                              # (N, 3) points on the unit sphere
triangles = dome.get_all_triangles().reshape(-1, 3)   # (F, 3) indices into xyz
print(f'stored vertices : {xyz.shape[0]}')
print(f'triangles       : {triangles.shape[0]}')
print(f'all on unit sphere? {np.allclose(np.linalg.norm(xyz, axis=1), 1.0)}')

stored vertices : 205
triangles       : 320
all on unit sphere? True


Vertices on the seams of the unfolded net (notebook 03) are stored more than once. `unique_mesh` from
`dome_utils` merges the copies into a clean mesh with 10 *f*² + 2 points:

In [6]:
points, faces, index_map = unique_mesh(dome)
print(f'unique vertices : {points.shape[0]}   (10 f^2 + 2 = {10 * dome.frequency ** 2 + 2})')
print(f'index_map maps each of the {len(index_map)} stored vertices to one of the {len(points)} points')

unique vertices : 162   (10 f^2 + 2 = 162)
index_map maps each of the 205 stored vertices to one of the 162 points


## 3. Individual vertices

A vertex is addressed by its `(x, y)` position on the integer index grid.

In [7]:
v = dome.get_vertex_at(6, 5)
print('vertex at grid (6, 5)')
print(f'  id            : {v.id}')
print(f'  xyz           : {np.round(v.coord, 4)}')
print(f'  (colat, lon)  : {np.round(v.latlon_coord, 4)}  radians, measured from the +y axis')
print(f'  seam copies   : {len(v.same_vertices or [])}')

corner = dome.get_vertex_at(0, 0)     # one of the 12 original icosahedron corners
print('\nvertex at grid (0, 0) is an icosahedron corner; its seam copies are at',
      [(s.x, s.y) for s in corner.same_vertices])

vertex at grid (6, 5)
  id            : 43
  xyz           : [ 0.9511  0.2764 -0.1382]
  (colat, lon)  : [1.2908 1.7151]  radians, measured from the +y axis
  seam copies   : 0

vertex at grid (0, 0) is an icosahedron corner; its seam copies are at [(20, 20)]


Any payload can be attached to a vertex, for example the weight vector of a SOM node (notebook 06):

In [8]:
v.set_data(np.random.default_rng(0).normal(size=3))
np.round(v.data, 3)

array([ 0.126, -0.132,  0.64 ])

### Try it: inspect any vertex

Move the slider through the stored vertices of the frequency-4 dome. Seam vertices list their copies,
and the 12 icosahedron corners have 5 neighbours instead of 6.

In [9]:
vertices = dome.get_all_vertices()


@interact(index=IntSlider(value=0, min=0, max=len(vertices) - 1, description='vertex'))
def inspect_vertex(index):
    u = vertices[index]
    dome.unmark_vertices()                         # always reset before a neighbour search
    neighbours = dome.get_neighbours(u, False)
    dome.unmark_vertices()
    print(f'grid (x, y)   : ({u.x}, {u.y})      id {u.id}')
    print(f'xyz           : {np.round(u.coord, 4)}')
    print(f'(colat, lon)  : {np.round(u.latlon_coord, 4)}')
    print(f'seam copies   : {[(s.x, s.y) for s in (u.same_vertices or [])]}')
    print(f'neighbours    : {len(neighbours)} at {[(n.x, n.y) for n in neighbours]}')

interactive(children=(IntSlider(value=0, description='vertex', max=204), Output()), _dom_classes=('widget-inte…

## 4. How the dome grows with frequency

In [10]:
print(' freq | stored | unique |  faces | mean edge (unit sphere)')
print('------+--------+--------+--------+------------------------')
for f in (1, 2, 3, 4, 6, 8, 12):
    d = GeodesicDome(f)
    p, fc, _ = unique_mesh(d)
    edges = np.linalg.norm(p[fc[:, 0]] - p[fc[:, 1]], axis=1)
    print(f' {f:4d} | {len(d.get_all_vertices()):6d} | {len(p):6d} | {len(fc):6d} | {edges.mean():.4f}')

 freq | stored | unique |  faces | mean edge (unit sphere)
------+--------+--------+--------+------------------------
    1 |     22 |     12 |     20 | 1.0515
    2 |     63 |     42 |     80 | 0.5823
    3 |    124 |     92 |    180 | 0.3953
    4 |    205 |    162 |    320 | 0.2983
    6 |    427 |    362 |    720 | 0.1998
    8 |    729 |    642 |   1280 | 0.1501
   12 |   1573 |   1442 |   2880 | 0.1002


### Try it: pick a frequency

In [11]:
@interact(f=IntSlider(value=8, min=1, max=32, description='frequency'))
def sizes(f):
    d = GeodesicDome(f)
    p, fc, _ = unique_mesh(d)
    edges = np.linalg.norm(p[fc[:, 0]] - p[fc[:, 1]], axis=1)
    print(f'stored vertices : {len(d.get_all_vertices())}')
    print(f'unique points   : {len(p)}   (10 f^2 + 2 = {10 * f * f + 2})')
    print(f'triangles       : {len(fc)}   (20 f^2 = {20 * f * f})')
    print(f'edge length     : mean {edges.mean():.4f}, min {edges.min():.4f}, max {edges.max():.4f} '
          f'(max/min = {edges.max() / edges.min():.3f})')

interactive(children=(IntSlider(value=8, description='frequency', max=32, min=1), Output()), _dom_classes=('wi…